In [0]:
%sql
use catalog my_first_pyspark_project;

In [0]:
df_customer = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_customers_dataset.csv")

In [0]:
df_geoloacation = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_geolocation_dataset.csv")

In [0]:
df_order_item = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_order_items_dataset.csv")

In [0]:
df_review = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_order_reviews_dataset.csv")

In [0]:
df_payment = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_order_payments_dataset.csv")

In [0]:
df_order = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_orders_dataset.csv")

In [0]:
df_product = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_products_dataset.csv")

In [0]:
df_seller = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_sellers_dataset.csv")

In [0]:
df_product_cat_translation = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/product_category_name_translation.csv")

In [0]:
from pyspark.sql.functions import *
def missing_value(df,df_name):
    print(f"Missing values in {df_name}:")
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).display()

In [0]:
missing_value(df_customer,'df_customer')

In [0]:
missing_value(df_order,'df_order')


In [0]:
missing_value(df_order_item,'df_order_item')


In [0]:
missing_value(df_payment,'df_payment')


In [0]:
# Handling Missing Value

df_order_clean = df_order.withColumn(
    "order_delivered_customer_date",
    when(
        col("order_delivered_customer_date").isNull(),
        to_timestamp(lit("1900-01-01T00:00:00.000+00:00"))
    ).otherwise(col("order_delivered_customer_date"))
)
df_order_clean.display()


In [0]:
df_payment.display()

In [0]:
from pyspark.sql.functions import *
mean = df_payment.select(avg(col("payment_value"))).collect()[0][0]
df_payment = df_payment.withColumn(
    "payment_value_mean",
    when(
        col("payment_value").isNull(),
        mean
    ).otherwise(col("payment_value"))
)
df_payment.display()

In [0]:
df_payment.printSchema()

In [0]:
df_payment_clean = df_payment.withColumn("payment_type", when(col("payment_type") == "credit_card", "credit_card").when(col("payment_type") == "debit_card", "debit_card").when(col("payment_type") == "boleto", "Bank transfer").otherwise("other"))

In [0]:
df_payment_clean.display()

In [0]:
df_customer.printSchema()



In [0]:
df_customer_clean = df_customer.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))

In [0]:
df_customer_clean.printSchema()

## Remove Duplicate Records

In [0]:
df_customer_clean = df_customer_clean.dropDuplicates(["customer_id"])

## Data Transformation

In [0]:
df_order_with_detail = df_order_clean.join(df_order_item,"order_id",'left')\
                       .join(df_payment_clean,"order_id",'left')\
                       .join(df_customer_clean,"customer_id",'left')

In [0]:
df_order_with_detail.display()

In [0]:
df_order_with_detail.groupBy("order_id").agg(round(sum("payment_value"),2).alias("total_order_value")).orderBy(col("total_order_value").desc()).display()

### Checking The Outliers

In [0]:
quantiles = df_order_item.approxQuantile("price",[0.01,0.99],0.0)
low_cutoff,high_cutoff = quantiles[0],quantiles[1]

In [0]:
low_cutoff,high_cutoff


In [0]:
df_order_item_clean = df_order_item.filter(col("price")>low_cutoff).filter(col("price")<high_cutoff)
df_order_item_clean.display()


In [0]:
df_order_item_clean.select("price").summary().display()


In [0]:
df_product_clean = df_product.withColumn("product_size_category", when(col("product_weight_g")<500,"small").when(col("product_weight_g").between(500,1000),"medium").otherwise("large"))
df_product_clean.display()

In [0]:
df_order_with_detail.printSchema()